In [1]:
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.datasets import RetrievalDataset, MassSpecDataset
from massspecgym.data.transforms import InMemCachedMolTransform
from chemembed_transforms import ChemEmbedSpecTransform, ChemEmbedMolTransform, ChemBERTaMolTransform, MoLFormerMolTransform
from pathlib import Path
from ann_model_massspecgym import AnnRetrieval
from pytorch_lightning import Trainer
from massspecgym.models.base import Stage
import pandas as pd
import sys
import os
import pickle
import tqdm as tqdm_module

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [ ]:
######### Embedding selection ##########

# Mol2Vec (300 dims)
# mol_transform_factory = lambda: ChemEmbedMolTransform(model_path='Mol2Vec_model/model_300dim.pkl')
# embedding_name = "mol2vec"
# mol_embedding_dim = 300

# ChemBERTa (768 dims)
mol_transform_factory = lambda: ChemBERTaMolTransform()
embedding_name = "chemberta"
mol_embedding_dim = 768

# MoLFormer (768 dims)
# mol_transform_factory = lambda: MoLFormerMolTransform()
# embedding_name = "molformer"
# mol_embedding_dim = 768

####################

selected_molecular_embedding = InMemCachedMolTransform(
    cache_pth=f"data/{embedding_name}_cache.pkl",
    mol_transform_factory=mol_transform_factory,
    verbose=True,
)

# Train dataset, without reference candidates
train_dataset = MassSpecDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)
data_module_train = MassSpecDataModule(dataset=train_dataset, batch_size=32)

data_module_train.prepare_data()
data_module_train.setup()

all_smiles = train_dataset.metadata["smiles"].tolist()
cache_path = Path(f"data/{embedding_name}_cache.pkl")

if sys.platform == "win32":
    # Windows: build sequentially
    if not cache_path.exists():
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        transform = mol_transform_factory()
        cache = {}
        for smi in tqdm_module.tqdm(sorted(set(all_smiles)), desc=f"Building {embedding_name} cache"):
            cache[smi] = transform.from_smiles(smi)
        with open(cache_path, "wb") as f:
            pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)
    selected_molecular_embedding._load_cache()
else:
    # Linux: use multiprocessing
    num_workers = max(1, os.cpu_count() - 1)
    selected_molecular_embedding.build_cache(all_smiles, num_workers=num_workers)

####################

# Init model
model = AnnRetrieval(
    mol_embedding_dim=mol_embedding_dim,
    log_only_loss_at_stages=[Stage.TRAIN, Stage.VAL]
)

# Init trainer
trainer = Trainer(
    accelerator="auto",
    max_epochs=15,
    log_every_n_steps=1
)

# Validation before training
trainer.validate(model, datamodule=data_module_train)

# Train
trainer.fit(model, datamodule=data_module_train)

# Save model checkpoint
trainer.save_checkpoint(f"ANN_trained_{embedding_name}.ckpt")

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Initializing InMemCachedMolTransform for ChemBERTaMolTransform. Cache file not provided or missing; molecules will be transformed on the fly. Call `build_cache(smiles_list)` to create it.


c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Building chemberta cache: 100%|██████████| 31602/31602 [1:22:19<00:00,  6.40it/s]  
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install l

Validation DataLoader 0:   9%|▊         | 52/608 [00:10<01:46,  5.20it/s]

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...

  | Name | Type      | Params
-----------------------------------
0 | ann  | ANN_Class | 146 M 
-----------------------------------
146 M     Trainable params
0         Non-trainable params
146 M     Total params
584.990   Total estimated model params size (MB)


Validation DataLoader 0:   9%|▊         | 52/608 [00:10<01:51,  4.97it/s]


c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0:   0%|          | 5/6067 [00:13<4:32:49,  0.37it/s, v_num=8, train_loss=0.979]

In [ ]:
# Standard test 
test_dataset = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)
data_module_test = MassSpecDataModule(dataset=test_dataset, batch_size=32)
data_module_test.setup(stage="test")

test_results = trainer.test(model, datamodule=data_module_test)
results_df = pd.DataFrame(test_results)
results_df.to_csv(f"masspecgym_ann_results_{embedding_name}.csv", index=False)

In [ ]:
# BONUS test with candidates (same molecular formula)
test_dataset_bonus = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
    candidates_pth='bonus'
)
data_module_test_bonus = MassSpecDataModule(dataset=test_dataset_bonus, batch_size=32)
data_module_test_bonus.setup(stage="test")

test_results_bonus = trainer.test(model, datamodule=data_module_test_bonus)
results_df_bonus = pd.DataFrame(test_results_bonus)
results_df_bonus.to_csv(f"masspecgym_ann_results_bonus_{embedding_name}.csv", index=False)